<a href="https://colab.research.google.com/github/ibrhmgldmr3/fly_kingo/blob/main/notebooks/colab_brain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# fly_kingo: sinek beyni hesapları (Colab, ücretsiz GPU)

Yerel makineyi yoran işleri burada çalıştırır: testler, karar yüzeyi, yüzey kontrolü, GRN ablasyonu.

**Başlamadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`.

Sonuçlar her batch'ten sonra Google Drive'a (`MyDrive/fly_kingo/`) yazılır. Colab bağlantısı koparsa
notebook'u baştan çalıştırmak yeterli; karar yüzeyi kaldığı yerden devam eder.

## 1. Ortam ve Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/fly_kingo")
(DRIVE / "flywire").mkdir(parents=True, exist_ok=True)
(DRIVE / "figures").mkdir(exist_ok=True)
print("Drive klasörü:", DRIVE)

## 2. Kod

İki yol var:
- **GitHub:** `BRANCH` dalı klonlanır (yerel değişiklikler push edilmiş olmalı).
- **Zip:** `BRANCH = None` bırakın; yerelde üretilen `fly_kingo_colab.zip` dosyasını yükleyin
  (veya Drive'da `MyDrive/fly_kingo/fly_kingo_colab.zip` olarak bırakın).

In [ ]:
BRANCH = "main"  # None ise zip kullanılır
REPO = "https://github.com/ibrhmgldmr3/fly_kingo.git"

import shutil, zipfile, os
ROOT = Path("/content/fly_kingo")
if ROOT.exists():
    shutil.rmtree(ROOT)

if BRANCH:
    !git clone --depth 1 --branch {BRANCH} {REPO} {ROOT}
else:
    bundle = DRIVE / "fly_kingo_colab.zip"
    if not bundle.exists():
        from google.colab import files
        uploaded = files.upload()  # fly_kingo_colab.zip seçin
        bundle = Path(next(iter(uploaded)))
    with zipfile.ZipFile(bundle) as z:
        z.extractall(ROOT)

os.chdir(ROOT)
!ls

## 3. Kurulum

In [ ]:
# Colab'da CUDA'lı torch zaten kurulu; pip onu değiştirmez.
!pip install -q -e ".[brain,dev]"
import torch; print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## 4. Connectome verisi (Drive'da önbelleklenir, checksum doğrulanır)

In [ ]:
data = ROOT / "data"
data.mkdir(exist_ok=True)
link = data / "flywire"
if not link.exists():
    link.symlink_to(DRIVE / "flywire")
!flyhire data download
!flyhire info

## 5. Testler

In [ ]:
!pytest -q

Yavaş (tam beyin) testleri CPU'da çalışır; Colab'ın 2 çekirdeğinde birkaç dakika sürebilir.

In [ ]:
!pytest -q -m slow

## 6. Karar yüzeyi (GPU)

Çıktı doğrudan Drive'a yazılır ve her batch'ten sonra kaydedilir. Yerelde yarım kalan
`data/surface_v783.npz` dosyanız varsa Drive'a `MyDrive/fly_kingo/surface_v783.npz` olarak koyun;
hesap oradan devam eder (dosya aynı beyin sürümüyle başlatılmış olmalı).

In [ ]:
SURFACE = DRIVE / "surface_v783.npz"
!flyhire brain surface --device cuda --batch-conditions 24 --out {SURFACE}

### Yüzey kontrolü ve görseller

In [ ]:
!flyhire brain check-surface --device cuda --points 24 --surface {SURFACE}
!flyhire brain heatmap --surface {SURFACE} --out {DRIVE}/figures/decision_surface.png
!flyhire brain heatmap --surface {SURFACE} --water 120 --out {DRIVE}/figures/decision_surface_water120.png

from IPython.display import Image, display
display(Image(str(DRIVE / "figures/decision_surface.png")))

## 7. (İsteğe bağlı) GRN ablasyonu

In [ ]:
!python scripts/grn_ablation.py --device cuda --out {DRIVE}/grn_ablation.json

## 8. Sonuçları yerele almak

Dosyalar Drive'da `MyDrive/fly_kingo/` altında. Yerelde `data/` ve `docs/figures/` içine kopyalayın.
İsterseniz buradan doğrudan indirin:

In [ ]:
from google.colab import files
files.download(str(SURFACE))